# Representation Compare Notebook

Live notebook workflow for `repr_analysis`.

This template does not require a precomputed batch directory. It can:
- run `analyze_repr.run_analysis()` directly for several checkpoints
- optionally save each per-model analysis directory
- build a compact key-metric table in memory
- save CSV / HTML tables and plot charts inline
- render PCA or t-SNE projection grids directly from the returned tensors
- run the full latent-geometry diagnostic suite from one unified entrypoint

The notebook imports `analyze_repr.py` and `run_full_diagnostics.py`.
Live analysis, table building, diagnostics, and plotting all come from repo modules.

Default metric focus in this template is now planning-first:
- one-step prediction quality
- autoregressive rollout drift
- expert-vs-random cost separation
- action sensitivity
- optional reference probe only when `STATE_KEY` is set
- robustness / resolution geometry from clean-vs-noisy latent shifts

In [ ]:
import os
from pathlib import Path
ROOT_DIR = Path(os.environ.get('PAPER1_DATA_ROOT', 'dataset/ag_data/data/world_model/quentinll'))


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'tools' / 'repr_analysis' / 'analyze_repr.py').exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.repr_analysis.analyze_repr import (
    DEFAULT_KEY_METRICS,
    build_metric_reference_table,
    build_key_metric_long_table_from_records,
    build_key_metric_table_from_records,
    format_analysis_report,
    make_unique_slug,
    plot_metric_bars,
    plot_metric_heatmap,
    plot_projection_grid_from_records,
    run_analysis,
    save_metric_reference_table,
    save_key_metric_tables,
    save_outputs,
)
from tools.repr_analysis.run_full_diagnostics import run_full_diagnostics


In [ ]:
# Edit these values for your current experiment set.
DATASET = str(ROOT_DIR / 'lewm-tworooms/tworoom')
STATE_KEY = None  # Optional external reference probe, e.g. 'proprio'
DEVICE = 'cuda'
INCLUDE_PLANNING = True  # Set False if your runtime checkout cannot run model.get_cost() reliably.

MODEL_SPECS = [
    (f'LeWM-base', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_lewm/tworoom_lewm_epoch_9_object.ckpt')),
    (f'LeWM-fixed-std', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_lewm_noise_std_0_005/tworoom_lewm_noise_std_0_005_epoch_9_object.ckpt')),
    (f'LeWM-perframe-p05', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_lewm_noise_0to005_p05/tworoom_lewm_noise_0to005_p05_epoch_9_object.ckpt')),
    (f'LeWM-perframe-p1', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_lewm_noise_0to005_p1/tworoom_lewm_noise_0to005_p1_epoch_9_object.ckpt')),
    (f'SWM-base', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_dim64_20260425/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_dim64_20260425_epoch_9_object.ckpt')),
    (f'SWM-fixed-std', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_std0_005_dim64/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_std0_005_dim64_epoch_9_object.ckpt')),
    (f'SWM-perframe-p05', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_0to005_p05_dim64/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_0to005_p05_dim64_epoch_9_object.ckpt')),
    (f'SWM-perframe-p1', str(ROOT_DIR / 'lewm-tworooms/ckpt/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_0to005_p1_dim64/tworoom_swm_mlp_bn_uniform_w02_t2_temporal_masked_2_noise_0to005_p1_dim64_epoch_9_object.ckpt')),
]

RUN_DIR = ROOT_DIR / 'lewm-pusht/repr_analysis/notebook_live_compare'
ANALYSIS_SAVE_DIR = RUN_DIR / 'analysis_runs'
EXPORT_DIR = RUN_DIR / 'exports'

EVAL_SCORES = {
    'LeWM-base': 93,
    'LeWM-fixed-std': 96,
    'LeWM-perframe-p05': 94,
    'LeWM-perframe-p1': 94,
    'SWM-base': 90.8,
    'SWM-fixed-std': 97.6,
    'SWM-perframe-p05': 87.3,
    'SWM-perframe-p1': 86.7,
}

MODEL_ORDER = [label for label, _ in MODEL_SPECS]
METRICS = DEFAULT_KEY_METRICS

# Live analysis settings passed to `run_analysis()`.
ANALYZE_KWARGS = dict(
    state_key=STATE_KEY,
    frameskip=5,
    img_size=224,
    n_sequences=128,
    future_steps=8,
    max_points=512,
    knn_k=10,
    action_trials=8,
    planning_random_trials=16,
    include_planning=INCLUDE_PLANNING,
    interp_steps=9,
    perturb_scale=1.0,
    seed=3072,
    device=DEVICE,
)

# Save/export settings passed to `save_outputs()` only.
SAVE_OUTPUT_KWARGS = dict(
    export_tsne=False,
    tsne_perplexity=30.0,
)

# Full latent-geometry diagnostics settings passed to `run_full_diagnostics()`.
# Keep NOISE_STDS aligned with eval.corruption.std sweeps.
NOISE_STDS = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]
NOISE_N_SEQUENCES = 256
NOISE_FUTURE_STEPS = 8
NOISE_FRAME_SCOPE = 'goal'  # 'goal' is the key diagnostic for noisy-goal eval drops.
NOISE_EMBEDDING_SPACE = None  # None uses each model's inference_cost_space.

# Predictor/task-resolution settings used by the same unified diagnostics call.
PREDICTOR_ROLLOUT_STEPS = [1, 2, 4, 8]
PREDICTOR_HISTORY_NOISE_ONLY = True  # mirrors pixels-only eval condition
SKIP_NOISE_DIAGNOSTIC = False
SKIP_PREDICTOR_DIAGNOSTIC = False
SKIP_RESOLUTION_DIAGNOSTIC = False
FULL_DIAGNOSTICS_DIR = EXPORT_DIR / 'full_diagnostics'


In [ ]:
# Live path: call analyze_repr.run_analysis() directly.
records = []
used_slugs = set()
if ANALYSIS_SAVE_DIR is not None:
    ANALYSIS_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for label, ckpt in MODEL_SPECS:
    slug = make_unique_slug(label, used_slugs)
    result, outputs = run_analysis(
        ckpt=ckpt,
        dataset=DATASET,
        log=print,
        **ANALYZE_KWARGS,
    )
    analysis_dir = ANALYSIS_SAVE_DIR / slug if ANALYSIS_SAVE_DIR is not None else None
    if analysis_dir is not None:
        save_outputs(
            analysis_dir,
            result,
            outputs['emb'],
            outputs.get('state'),
            export_tsne=SAVE_OUTPUT_KWARGS.get('export_tsne', False),
            tsne_perplexity=SAVE_OUTPUT_KWARGS.get('tsne_perplexity', 30.0),
            seed=ANALYZE_KWARGS.get('seed', 3072),
        )
    records.append({
        'label': label,
        'slug': slug,
        'analysis_dir': str(analysis_dir) if analysis_dir is not None else '',
        'result': result,
        'outputs': outputs,
        'report': format_analysis_report(result),
    })

print(records[0]['report'])

In [ ]:
key_wide = build_key_metric_table_from_records(
    records,
    metrics=METRICS,
    eval_scores=EVAL_SCORES,
    model_order=MODEL_ORDER,
    include_analysis_dir=True,
)

key_long = build_key_metric_long_table_from_records(
    records,
    metrics=METRICS,
    eval_scores=EVAL_SCORES,
    model_order=MODEL_ORDER,
)

display(key_wide)
save_paths = save_key_metric_tables(key_wide, key_long, output_dir=EXPORT_DIR, stem='key_metrics')
save_paths

## Full latent-geometry diagnostics

Run the complete diagnostic suite through a single entrypoint: `run_full_diagnostics()`.
It executes and saves:

- encoder noise sensitivity: clean/noisy angular shift, nearest-neighbor geometry, robust radius, CKA;
- predictor sensitivity: noisy-vs-clean history rollout drift;
- task resolution: transition-resolution ratio, inverse-dynamics linear probe, LiDAR / effective rank;
- one-row `diagnostics_summary` for P0 correlation / active validation.

Reference mapping follows `research_notebook_swm.md` §7: RankMe / LiDAR for rank, Wang & Isola for sphere uniformity geometry, KNN-OOD / SNGP for nearest-neighbor distance, Lipschitz / randomized smoothing / Jacobian probing for input-noise shift, CKA for subspace alignment, Dreamer / TD-MPC for rollout-error primitives, and Brandfonbrener et al. for inverse-dynamics representation probing.


In [ ]:
# Full latent-geometry diagnostics: one call runs noise sensitivity,
# predictor sensitivity, task resolution, roll-up summaries, and plots.
diagnostics = run_full_diagnostics(
    models={label: ckpt for label, ckpt in MODEL_SPECS},
    dataset=DATASET,
    stds=NOISE_STDS,
    rollout_steps=PREDICTOR_ROLLOUT_STEPS,
    state_key=STATE_KEY,
    n_sequences=NOISE_N_SEQUENCES,
    future_steps=max(NOISE_FUTURE_STEPS, max(PREDICTOR_ROLLOUT_STEPS)),
    frameskip=ANALYZE_KWARGS.get('frameskip', 1),
    img_size=ANALYZE_KWARGS.get('img_size', 224),
    embedding_space=NOISE_EMBEDDING_SPACE,
    predictor_history_noise_only=PREDICTOR_HISTORY_NOISE_ONLY,
    skip_noise=SKIP_NOISE_DIAGNOSTIC,
    skip_predictor=SKIP_PREDICTOR_DIAGNOSTIC,
    skip_resolution=SKIP_RESOLUTION_DIAGNOSTIC,
    seed=ANALYZE_KWARGS.get('seed', 3072),
    device=DEVICE,
    save_dir=FULL_DIAGNOSTICS_DIR,
    plot=True,
    log=print,
)

noise_rows = diagnostics['noise_rows']
geometry_summary = diagnostics['geometry_summary']
predictor_rows = diagnostics['predictor_rows']
resolution_rows = diagnostics['resolution_rows']
noise_table = diagnostics['noise_table']
predictor_table = diagnostics['predictor_table']
resolution_table = diagnostics['resolution_table']
diagnostics_summary = pd.DataFrame(diagnostics['diagnostics_summary'])

if noise_table is not None:
    display(noise_table)
if geometry_summary is not None:
    display(geometry_summary)
if predictor_table is not None:
    display(predictor_table)
if resolution_table is not None:
    display(resolution_table)
display(diagnostics_summary)

FULL_DIAGNOSTICS_DIR


## Predictor sensitivity

Single-step open-loop target shift + autoregressive rollout drift between
noisy- and clean-history conditioning. Captures how much pixel noise on the
predictor's input gets amplified through one or several rollout steps.

References: Dreamer / TD-MPC2 single-step rollout MSE; multi-step
noisy-vs-clean drift is novel here (see research_notebook_swm.md §7).


In [ ]:
# Predictor sensitivity table from the unified diagnostics call above.
# Re-run the full diagnostics cell if you changed NOISE_STDS / MODEL_SPECS.
if predictor_table is None:
    print('predictor diagnostics were skipped')
else:
    display(predictor_table)
    predictor_table.to_csv(EXPORT_DIR / 'predictor_sensitivity.csv', index=False)


In [ ]:
# Plot rollout drift from the unified diagnostics output.
if predictor_rows:
    df_pred = pd.DataFrame(predictor_rows)
    fig, axes = plt.subplots(1, len(PREDICTOR_ROLLOUT_STEPS),
                              figsize=(4 * len(PREDICTOR_ROLLOUT_STEPS), 3.5),
                              sharey=False)
    if len(PREDICTOR_ROLLOUT_STEPS) == 1:
        axes = [axes]
    for ax, T in zip(axes, PREDICTOR_ROLLOUT_STEPS):
        col = f'rollout_T{T}_l2_median'
        if col not in df_pred.columns:
            continue
        for label, group in df_pred.groupby('model', sort=False):
            group = group.sort_values('std')
            ax.plot(group['std'], group[col], marker='o', label=str(label))
        ax.set_title(f'rollout T={T}')
        ax.set_xlabel('pixel noise std')
        ax.set_ylabel('latent L2 drift')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(EXPORT_DIR / 'predictor_rollout_drift.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('predictor diagnostics were skipped')


## Task resolution

Quantifies whether the encoder preserves fine-grained transitions a
controller needs. Two indicators:

- `transition_resolution_ratio`: median consecutive-pair distance over
  median random cross-sequence pair distance. Smaller = better separation.
- Inverse-dynamics linear probe R^2: closed-form ridge regression from
  `concat(z_t, z_{t+1})` to `action_t`. Justified as a resolution proxy
  by Brandfonbrener et al. (NeurIPS 2023).

Plus `lidar_rank` (Thilak et al., ICLR 2024) and `clean_effective_rank`
(RankMe; Garrido et al., ICML 2023) as orthogonal collapse / clustering
diagnostics.


In [ ]:
# Task-resolution table from the unified diagnostics call above.
if resolution_table is None:
    print('task-resolution diagnostics were skipped')
else:
    display(resolution_table)
    resolution_table.to_csv(EXPORT_DIR / 'task_resolution.csv', index=False)


## Diagnostics roll-up

Combine encoder shift / NN geometry / predictor drift / task resolution
into one row per model. This is the table fed into the P0.4 correlation
analysis (Spearman ρ + bootstrap CI vs eval drop).


In [ ]:
# Per-model roll-up from the unified diagnostics call above.
# This mirrors `diagnostics_summary.json` and feeds P0 correlation analysis.
if EVAL_SCORES and not diagnostics_summary.empty:
    diagnostics_summary['eval_score'] = diagnostics_summary['model'].map(EVAL_SCORES)
display(diagnostics_summary)
diagnostics_summary.to_csv(EXPORT_DIR / 'diagnostics_summary.csv', index=False)


In [ ]:
metric_reference = build_metric_reference_table(METRICS)
display(metric_reference)
reference_paths = save_metric_reference_table(metric_reference, output_dir=EXPORT_DIR)
reference_paths

In [ ]:
plot_metric_bars(
    key_wide,
    metrics=METRICS,
    output=EXPORT_DIR / 'key_metrics_bars.png',
    ncols=3,
);

In [ ]:
plot_metric_heatmap(
    key_wide,
    metrics=METRICS,
    output=EXPORT_DIR / 'key_metrics_heatmap.png',
);

In [ ]:
plot_projection_grid_from_records(
    records,
    model_labels=MODEL_ORDER,
    projection='pca',
    color_dims=[0, 1],
    output=EXPORT_DIR / 'pca_projection_grid.png',
);

If you only want to compare saved live-analysis records, keep the `records` list and rerun the table / plotting cells without recomputing the model outputs.